# Phase 4: Modelling (Part 3 - Hyperparameter Tuning)
**Client:** Crédit Nationale Azur

**Objective:** Use GridSearchCV to optimise the parameters of our Random Forest and Support Vector Machine models to maximise the F1-Score.

In this notebook, we will recreate our strict data preparation pipeline to ensure a clean environment, then deploy systematic grid searches across both algorithms.

In [ ]:
# 1. Imports
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

# 2. Load and Map Target
df = joblib.load('../data/cleaned_data.pkl')
df['personal_loan'] = df['personal_loan'].map({'yes': 1, 'no': 0})
X = df.drop(['personal_loan', 'customer_id'], axis=1)
y = df['personal_loan']

# 3. Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_train = X_train.copy()
X_test = X_test.copy()

# 4. Encode Text Variables
categorical_cols = ['education_level', 'credit_card_acct', 'online_acct']
for col in categorical_cols:
    X_train = pd.concat([X_train, pd.get_dummies(X_train[col], prefix=col, dtype=int)], axis=1).drop([col], axis=1)
    X_test = pd.concat([X_test, pd.get_dummies(X_test[col], prefix=col, dtype=int)], axis=1).drop([col], axis=1)
X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)

# 5. Feature Selection
selector = SelectKBest(score_func=chi2, k=7)
X_train_selected_array = selector.fit_transform(X_train, y_train)
X_test_selected_array = selector.transform(X_test)
selected_features = X_train.columns[selector.get_support()]
X_train_selected = pd.DataFrame(X_train_selected_array, columns=selected_features, index=X_train.index)
X_test_selected = pd.DataFrame(X_test_selected_array, columns=selected_features, index=X_test.index)

# 6. Standardisation
continuous_features = ['age', 'yrs_experience', 'family_size', 'income', 'mortgage_amt', 'credit_card_spend']
cols_to_scale = [col for col in continuous_features if col in selected_features]
for col in cols_to_scale:
    scaler = StandardScaler()
    X_train_selected[col] = scaler.fit_transform(X_train_selected[col].values.reshape(-1, 1)).flatten()
    X_test_selected[col] = scaler.transform(X_test_selected[col].values.reshape(-1, 1)).flatten()

print("Data prepared for GridSearchCV.")

Data rigorously prepared for GridSearchCV.


## 1. Tuning the Random Forest
We will test different numbers of trees (`n_estimators`), tree depths (`max_depth`), and split criteria (`criterion`).

In [2]:
# Define the parameter grid for Random Forest
rf_param_grid = [
    {
        'n_estimators': [50, 100, 200],
        'max_depth': [None, 10, 20],
        'criterion': ['gini', 'entropy']
    }
]

# Instantiate GridSearchCV using cv=5 and scoring='f1'
rf_grid = GridSearchCV(
    RandomForestClassifier(random_state=42), 
    rf_param_grid, 
    cv=5, 
    scoring='f1', 
    verbose=1
)

# Fit the grid
print("Tuning Random Forest...")
rf_grid.fit(X_train_selected, y_train)

print("\nRandom Forest Tuning Complete.")
print(f"Best Parameters: {rf_grid.best_params_}")
print(f"Best Cross-Validated F1-Score: {rf_grid.best_score_:.4f}")

Tuning Random Forest...
Fitting 5 folds for each of 18 candidates, totalling 90 fits

Random Forest Tuning Complete.
Best Parameters: {'criterion': 'gini', 'max_depth': 10, 'n_estimators': 100}
Best Cross-Validated F1-Score: 0.7133


## 2. Tuning the Support Vector Machine (SVM)
As per our class requirements, we will use `np.arange` to generate float ranges for the `C` parameter (regularisation). We will also test different kernel types and class weight penalties to handle the imbalance.

In [3]:
# Define the parameter grid for SVM using np.arange for float ranges
svm_param_grid = [
    {
        'C': np.arange(0.5, 2.5, 0.5), # Tests 0.5, 1.0, 1.5, 2.0
        'kernel': ['linear', 'rbf'],
        'class_weight': [None, 'balanced']
    }
]

# Instantiate GridSearchCV using cv=5 and scoring='f1'
svm_grid = GridSearchCV(
    SVC(random_state=42), 
    svm_param_grid, 
    cv=5, 
    scoring='f1', 
    verbose=1
)

# Fit the grid
print("Tuning Support Vector Machine...")
svm_grid.fit(X_train_selected, y_train)

print("\nSVM Tuning Complete.")
print(f"Best Parameters: {svm_grid.best_params_}")
print(f"Best Cross-Validated F1-Score: {svm_grid.best_score_:.4f}")

Tuning Support Vector Machine...
Fitting 5 folds for each of 16 candidates, totalling 80 fits

SVM Tuning Complete.
Best Parameters: {'C': np.float64(2.0), 'class_weight': None, 'kernel': 'rbf'}
Best Cross-Validated F1-Score: 0.6988
